<a href="https://colab.research.google.com/github/shuyu-d/IDL_TPs/blob/master/tp6_idl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 6 - Décente de gradient stochastique (SGD) pour l'entraînement d'un MLP 

Dans ce TP, nous implémentons les fonctions pour le calcul du passe avant et du passe arrière pour entraîner un MLP avec l'algorithme SGD. 

Les données d'observation appartiennent au jeu de données Iris, déjà disponible via scikit-learn (`sklearn datasets`, voir "Préparation du jeu de données"). Plus d'information sur ce jeu de données : https://scikit-learn.org/1.4/auto_examples/datasets/plot_iris_dataset.html,   https://archive.ics.uci.edu/dataset/53/iris. 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

### Préparation du jeu de données 
Ce jeu de données contient trois espèces différentes d’iris (Setosa, Versicolour et Virginica), caractérisées par la longueur et la largeur des sépales et des pétales. 
Les données sont stockées dans un tableau NumPy de dimension 150 x 4.

Les lignes correspondent aux observations (échantillons) et les colonnes représentent respectivement : longueur du sépale, largeur du sépale, longueur du pétale et largeur du pétale.

In [ ]:
iris = load_iris()
X = iris.data          # shape: (150, 4)
y = iris.target        # shape: (150,) ; classes 0,1,2


n_samples, n_features = X.shape
n_classes = len(np.unique(y))

print("X shape:", X.shape)
print("Nombre de classes:", n_classes)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Standardisation
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Classification multiclasses  

On définit $X$ le vecteur aléatoire associé aux quatres charactéristiques dans le jeu de données Iris. Maintenant l'ensemble $\mathcal{D} = (x^{(i)}, y^{(i)})_{i=1}^n$ avec $x^{(i)}\in\mathbb{R}^4$ et $y^{(i)} \in \{0,1,2\}$. 


Encodage one-hot pour les labels des trois classes : 

In [ ]:
# Encodage one-hot
def one_hot(y, num_classes):
    Y = np.zeros((len(y), num_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

Y_train = one_hot(y_train, n_classes)
Y_test = one_hot(y_test, n_classes) 

In [ ]:

n, p = X_train.shape

print('A few rows of Xtrain (shape %d x %d) : \n' %(n,p), pd.DataFrame(X_train).head() ) 
print('A few rows of y_train (shape %d x 1): \n'%y_train.shape[0], pd.DataFrame(Y_train).head() ) 

###  Construction et initialisation d'un MLP 

**Parametres du MLP** 

* Entrées : $p=4$ 
* 1 couche cachée : $\sigma_1=$ReLU
    * couche cachée 1: 10 neurones
    
* Sortie : 
    * $\sigma_2$=softmax 
    * $y=f_{\theta}(x) \in (0,1)^{K}$ où $K=3$.  
  
**Risque empirique** 

Pour un seul échantillon $(x, y)$, la fonction de perte du MLP (paramétré par $\theta=(W^{(i)},b^{(i)})_{i=1,2}$) est définie par l'entropie croisée (cross entropy) : 
    $L(\hat{y}, y) = - \sum_{k=1}^K y_k \log( \hat{y}_k ),$
où $\hat{y}$ est la sortie du MLP ($\theta$) avec l'entrée $x$. 

On définit le risque empirique suivant comme l'objectif de l'entraînement : 
$$ R(\theta; \mathcal{D}) = \frac{1}{2}\sum_{i=1}^n L(\hat{y}^{(i)}, y^{(i)} ),$$
où $\hat{y}^{(i)}$ est la sortie du MLP ($\theta$) avec l'entrée $x^{(i)}$. 


In [ ]:
# Parametres du MLP (2 couches cachées : sigma_1=ReLU, sigma_2=ReLU, sortie sigma_3=sigmoid)
#  - couche cachée 1: 10 neurones
#  - Sorite:          label one-hot de dimension 3 

# Fonctions utilitaires

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def softmax(z):
    """
    z: vecteur de taille (n_classes,)
    """
    z_shift = z - np.max(z)  # stabilité numérique
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z)

def cross_entropy_loss(y_true, y_pred):
    """
    y_true: vecteur one-hot (n_classes,)
    y_pred: probabilités softmax (n_classes,)
    """
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.sum(y_true * np.log(y_pred))

def accuracy(y_true_labels, y_pred_labels):
    return np.mean(y_true_labels == y_pred_labels)


# Architecture du MLP
#    Entrée (4) -> Cachée (H) -> Sortie (3)

input_dim = n_features      # 4
hidden_dim = 10             # H = nombre de neurones cachés
output_dim = n_classes      # 3

rng = np.random.default_rng(42)

# Initialisation des poids
W1 = rng.normal(0, 0.5, size=(hidden_dim, input_dim))   # (H, 4)
b1 = np.zeros(hidden_dim)                               # (H,)

W2 = rng.normal(0, 0.5, size=(output_dim, hidden_dim)) # (3, H)
b2 = np.zeros(output_dim)                               # (3,)

### Question (a) : passe avant

Compléter la fonction `passe_avant()` pour l'évaluation de la fonction de prédiction $f_{\theta}(\cdot)$ sur un seul échantillon $(x,y)\in\mathcal{D}$, étant donné le paramètre $\theta=(W^{(i)}, b^{(i)})_{i=1,2}$ du MLP. 



In [ ]:
# Passe avant (forward pass) pour un seul échantillon (x,y)

def forward(x, W1, b1, W2, b2):
    """
    x : shape (4,)
    Retourne toutes les quantités utiles pour la rétropropagation
    """
    # Couche cachée
    z1 = #--A COMPLÉTER----#            # (H,)
    a1 = #--A COMPLÉTER----#            # (H,)

    # Couche de sortie
    z2 = #--A COMPLÉTER----#           # (3,)
    y_hat = #--A COMPLÉTER----#        # (3,)

    cache = {
        "x": x,
        "z1": z1,
        "a1": a1,
        "z2": z2,
        "y_hat": y_hat
    }
    return y_hat, cache

# Fonctions pour la prédiction f_{\theta}(x) et pour l'évaluation: 

def predict(X, W1, b1, W2, b2):
    preds = []
    for x in X:
        y_hat, _ = forward(x, W1, b1, W2, b2)
        preds.append(np.argmax(y_hat))
    return np.array(preds)

def evaluate(X, y_labels, Y_onehot, W1, b1, W2, b2):
    total_loss = 0.0
    preds = []

    for x, y_true_vec in zip(X, Y_onehot):
        y_hat, _ = forward(x, W1, b1, W2, b2)
        total_loss += cross_entropy_loss(y_true_vec, y_hat)
        preds.append(np.argmax(y_hat))

    preds = np.array(preds)
    avg_loss = total_loss / len(X)
    acc = accuracy(y_labels, preds)
    return avg_loss, acc




### Passe arrière 
A l'aide des équations pour le calcul des dérivées partielles $\frac{\partial R}{\partial W^{(\ell)}_{ij}}$, compléter les lignes pour le passe arrière dans le bloc de l'algorithme GD suivant. 

**Question (b)**

Pour un seul échantillon $D'=(x,y)$, la fonction loss du SGD est $R(\theta,D')= L(\hat{y}, y)$ où $\hat{y}$ est la sortie du MLP avec l'entrée $x$. 

(i) Montrer que la dérivée $\frac{\partial L}{\partial\hat{y}_k}= -y_k\frac{1}{\hat{y}_k}$ for $k=1,2,3$. 

(ii) Pour la fonction softmax à la sortie $\hat{y}=\text{softmax}(z^{(2)})$, on se donne les dérivées partielles, pour tout $1\leq j,k \leq K=3$ : $\frac{\partial \hat{y}_j}{\partial z^{(2)}_k} = \hat{y}_j(\delta_{jk} - \hat{y}_k)$, où $\delta_{jk}=1$ si $j=k$ et $\delta_{jk}=0$ si $j\neq k$. Ensuite montrer que la dérivée $\frac{\partial L}{\partial z^{(2)}_k} = \hat{y}_k - y_k$. 

In [ ]:
# Backpropagation pour un seul échantillon 

def backward(y_true, cache, W2):
    """
    y_true : shape (3,) (one-hot)
    cache  : sorite, les pré-activations (z) et les activations (a) issu de forward
    W2     : utile pour rétropropager vers la couche cachée

    Retourne les gradients :
      dW1, db1, dW2, db2
      
    """
    x = cache["x"]         # (4,)
    z1 = cache["z1"]       # (H,)
    a1 = cache["a1"]       # (H,)
    y_hat = cache["y_hat"] # (3,)

    # --------------------------------------------------------
    # Sortie softmax -> fonction loss R(theta): cross-entropy 
    # delta2 = dR/dz2 
    # --------------------------------------------------------

    delta2 = #--A COMPLÉTER----#       # (3,)

    dW2 = #--A COMPLÉTER----#          # (3, H) où H est le nombre de neurones de la couche cachée 
    db2 = delta2                       # (3,)

    # --------------------------------------------------------
    # Couche cachée :
    # delta1 = (W2^T delta2) ⊙ ReLU'(z1)
    # --------------------------------------------------------
    delta1 = (W2.T @ delta2) * relu_derivative(z1)   # (H,)

    dW1 = np.outer(delta1, x)          # (H, 4)
    db1 = delta1                       # (H,)

    return dW1, db1, dW2, db2



### Entraînement avec SGD 

In [ ]:
# 8) Entraînement avec SGD
# ============================================================

learning_rate = 0.01
n_epochs = 300

train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
times_epo = []
t1 = time.time()
for epoch in range(n_epochs):
    # Mélange des données à chaque époque
    indices = np.arange(len(X_train))
    np.random.shuffle(indices)

    X_train_shuffled = X_train[indices]
    Y_train_shuffled = Y_train[indices]

    # --------------------------------------------------------
    # SGD : une mise à jour par exemple
    # --------------------------------------------------------
    for x, y_true in zip(X_train_shuffled, Y_train_shuffled):
        # Forward
        y_hat, cache = forward(x, W1, b1, W2, b2)

        # Backward
        dW1, db1, dW2, db2 = backward(y_true, cache, W2)

        # Mise à jour SGD
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    times_epo.append( time.time()-t1 )
    # Évaluation à la fin de l'époque
    train_loss, train_acc = evaluate(X_train, y_train, Y_train, W1, b1, W2, b2)
    test_loss, test_acc = evaluate(X_test, y_test, Y_test, W1, b1, W2, b2)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | "
              f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | "
              f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

#### Résultats 

In [ ]:
# Résultats finaux

y_pred_test = predict(X_test, W1, b1, W2, b2)

print("\n=== Accuracy ===")
print("Accuracy train :", train_accuracies[-1])
print("Accuracy test  :", test_accuracies[-1])

print("\n Prédictions sur quelques données test :")
for i in range(min(10, len(X_test))):
    print(f"Vrai = {y_test[i]}, Prédit = {y_pred_test[i]}")


# ----
# Courbes d'apprentissage
import matplotlib.pyplot as plt
epochs = np.arange(1, n_epochs + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, label="Train loss")
# plt.plot(times_epo, train_losses, label="Train loss")
plt.plot(epochs, test_losses, label="Test loss")
# plt.plot(times_epo, test_losses, label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.grid(True)